# Homophily and Structural Balance

**Lecture:** NS05 — Homophily and Structural Balance  
**Reading:** Easley & Kleinberg, Chapters 4 and 5

## Learning objectives

By the end of this notebook you will be able to:

1. Apply the **homophily test** ($2pq$ baseline) to detect homophily from node attributes.
2. Compute and plot **$k_{nn}(k)$** to classify a network as assortative, neutral, or disassortative.
3. Interpret the **degree assortativity coefficient** $r$.
4. Simulate the **Schelling model** and observe how local preferences produce global segregation.

In [ ]:
from netsci_utils import *            # nx, np, plt; set_seeds(); setup_matplotlib()
import matplotlib.colors as mcolors   # needed for Schelling grid colormap
import pandas as pd
from collections import defaultdict

set_seeds()        # RANDOM_SEED = 42
%matplotlib inline

---
## Part 1 — Homophily Test

### 1.1 Theoretical background

> **Homophily test (NS05):** Let $p$ and $q = 1 - p$ be the fractions of the two node types.
> Under a null model where edges form independently of type, the expected fraction of cross-type edges is $2pq$.
>
> - If the fraction of cross-type edges $\ll 2pq$ → **homophily**.
> - If the fraction of cross-type edges $\gg 2pq$ → **inverse homophily**.

This matches the test illustrated in NS05 slides:  
$\frac{5}{18} < \frac{4}{9} \Rightarrow \text{homophily}$

### 1.2 Worked example — reproduce the slide

We construct the exact small network from the NS05 slides to verify the formula.

In [ ]:
# Small example from NS05 slides: 9 nodes, 18 edges, 6 white / 3 pink nodes
# p = 6/9 = 2/3 (white), q = 3/9 = 1/3 (pink)
# 5 cross-type edges → fraction = 5/18
# 2pq = 2*(2/3)*(1/3) = 4/9

p_slide = 6/9
q_slide = 1 - p_slide
baseline_slide = 2 * p_slide * q_slide
frac_cross_slide = 5/18

print(f"p = {p_slide:.3f},  q = {q_slide:.3f}")
print(f"2pq (baseline)         = {baseline_slide:.4f}  ({4}/{9} = {4/9:.4f})")
print(f"Fraction cross edges   = {frac_cross_slide:.4f}  ({5}/{18})")
print(f"Homophily signal: {frac_cross_slide:.4f} < {baseline_slide:.4f}  → {frac_cross_slide < baseline_slide}")

### 1.3 Karate Club — faction homophily

In [ ]:
G = nx.karate_club_graph()
clubs = nx.get_node_attributes(G, 'club')  # 'Mr. Hi' or 'Officer'

types = list(clubs.values())
N = len(types)

# p = fraction of 'Mr. Hi', q = fraction of 'Officer'
p = types.count('Mr. Hi') / N
q = 1 - p
baseline = 2 * p * q    # expected cross-type fraction under no homophily   [NS05]

# Count cross-type edges
cross  = sum(1 for u, v in G.edges() if clubs[u] != clubs[v])
frac_cross = cross / G.number_of_edges()

print(f"Factions — Mr. Hi: {types.count('Mr. Hi')}, Officer: {types.count('Officer')}")
print(f"p = {p:.3f},  q = {q:.3f}")
print(f"2pq (baseline)       = {baseline:.4f}")
print(f"Fraction cross edges = {frac_cross:.4f}  ({cross}/{G.number_of_edges()})")
print(f"Homophily detected:  {frac_cross < baseline}")

In [ ]:
# Richer view: attribute mixing matrix
mixing = nx.attribute_mixing_matrix(G, 'club', normalized=True)

fig, ax = plt.subplots(figsize=FIGURE_SIZE_SMALL)
im = ax.imshow(mixing, cmap='Blues', vmin=0, vmax=0.5)
labels = ['Mr. Hi', 'Officer']
ax.set_xticks([0, 1]); ax.set_xticklabels(labels)
ax.set_yticks([0, 1]); ax.set_yticklabels(labels)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{mixing[i,j]:.2f}", ha='center', va='center', fontsize=12)
ax.set_title("Attribute mixing matrix\n(fraction of edges)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

**Observation:** The diagonal values (within-faction edges) are much larger than the off-diagonal (cross-faction), confirming strong homophily in the Karate Club.

### 1.4 Exercise

**Exercise 1 (guided):** Repeat the homophily test using the node **degree class** as the attribute (low-degree: $k < 5$, high-degree: $k \geq 5$).  
Is the network assortative by degree (do high-degree nodes preferentially connect to each other)?

In [ ]:
# Exercise 1: your solution here
degree_class = {n: ('high' if G.degree(n) >= 5 else 'low') for n in G.nodes()}

types_deg = list(degree_class.values())
p_deg = types_deg.count('high') / len(types_deg)
q_deg = 1 - p_deg
baseline_deg = 2 * p_deg * q_deg

cross_deg = sum(1 for u, v in G.edges() if degree_class[u] != degree_class[v])
frac_cross_deg = cross_deg / G.number_of_edges()

print(f"Degree-class homophily test:")
print(f"  p={p_deg:.3f}, q={q_deg:.3f}, 2pq={baseline_deg:.4f}")
print(f"  Fraction cross-class edges = {frac_cross_deg:.4f}")
print(f"  Assortativity by degree: {frac_cross_deg < baseline_deg}")

---
## Part 2 — Degree Assortativity and $k_{nn}(k)$

### 2.1 Theoretical background

> **Average neighbor degree (NS05):**
> $$k_{nn}(i) = \frac{1}{k_i} \sum_j A_{ij} k_j$$
> This is the average degree of node $i$'s neighbors.

> **Degree correlation function (NS05):**
> $$k_{nn}(k) = \langle k_{nn}(i) \rangle_{i:\, k_i = k}$$
> Plot $(k,\, k_{nn}(k))$ to classify the network:
> - **Increasing** → assortative (hubs connect to hubs, e.g. social networks)
> - **Flat** → neutral (no degree correlation)
> - **Decreasing** → disassortative (hubs connect to low-degree nodes, e.g. biological networks)

### 2.2 Worked example — manual computation

In [ ]:
# Small example from NS05 slides
# Nodes 0,1,2,3 with degrees 3,1,2,2
# k_nn(0) = (2+2+1)/3 = 5/3, k_nn(1) = 3, k_nn(2) = 5/2, k_nn(3) = 5/2

ex = nx.Graph([(0,1),(0,2),(0,3),(1,2),(2,3)])   # reproduces ns05 slide example
k_nn_i = nx.average_neighbor_degree(ex)           # k_nn(i) = 1/k_i * sum A_ij*k_j  [NS05]

for node in sorted(ex.nodes()):
    print(f"Node {node}: degree={ex.degree(node)},  k_nn(i)={k_nn_i[node]:.4f}")

print("\nExpected from NS05 slides:")
print("  k_nn(0) = 5/3 =", round(5/3, 4))
print("  k_nn(1) = 3")
print("  k_nn(2) = 5/2 =", round(5/2, 4))
print("  k_nn(3) = 5/2 =", round(5/2, 4))

### 2.3 Three network types — reproducing the NS05 slide plots

In [ ]:
def knn_k(H):
    """Compute k_nn(k): average k_nn(i) grouped by degree k.   [NS05]"""
    k_nn_i = nx.average_neighbor_degree(H)
    groups = defaultdict(list)
    for node, val in k_nn_i.items():
        groups[H.degree(node)].append(val)
    ks    = sorted(groups.keys())
    means = [np.mean(groups[k]) for k in ks]
    return ks, means

networks = {
    "Karate Club\n(social — assortative)": nx.karate_club_graph(),
    "Watts–Strogatz\n(regular rewired — neutral)": nx.watts_strogatz_graph(200, 6, 0.05, seed=RANDOM_SEED),
    "Barabási–Albert\n(scale-free — disassortative)": nx.barabasi_albert_graph(200, 2, seed=RANDOM_SEED),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = ['steelblue', 'seagreen', 'coral']

for ax, (title, H), color in zip(axes, networks.items(), colors):
    ks, means = knn_k(H)
    ax.scatter(ks, means, s=35, color=color, alpha=0.8)
    # Trend line
    z = np.polyfit(ks, means, 1)
    xline = np.linspace(min(ks), max(ks), 100)
    ax.plot(xline, np.polyval(z, xline), '--', color='#555555', linewidth=1)
    r = nx.degree_assortativity_coefficient(H)
    ax.set_xlabel("Degree $k$")
    ax.set_ylabel("$k_{nn}(k)$")
    ax.set_title(f"{title}\n$r = {r:.3f}$")
    ax.grid(alpha=0.3)

plt.suptitle("Degree correlation function $k_{nn}(k)$ for three network types [NS05]",
             y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

**Observation:** The slope of $(k, k_{nn}(k))$ directly mirrors the three cases in the NS05 slides:
- **Karate Club** ($r > 0$): increasing — hubs connect to other hubs (core–periphery structure).
- **Watts–Strogatz** ($r \approx 0$): flat — no degree correlation.
- **Barabási–Albert** ($r < 0$): decreasing — hubs connect to low-degree leaves (star-like structure).

### 2.4 Exercise

**Exercise 2 (open):** The NS05 slides show three real-world examples: (1) scientific collaboration network (assortative), (2) power grid (neutral), (3) metabolic network (disassortative). Load three equivalent NetworkX built-in graphs and:
1. Compute $r$ with `nx.degree_assortativity_coefficient()`.
2. Plot $k_{nn}(k)$ and confirm the slope matches the expected category.
3. Report which real network each most resembles.

In [ ]:
# Exercise 2: your solution here
# Hint: try nx.les_miserables_graph(), nx.petersen_graph(), nx.florentine_families_graph()

---
## Part 3 — Schelling Segregation Model

### 3.1 Theoretical background

> **Schelling model (NS05):** Agents of two types occupy a grid. Each agent is *satisfied* if at least $t$ of its 8 neighbors share its type. Unsatisfied agents move to a random empty cell.
>
> **Key insight:** Even when $t < 4$ (agents are happy being a minority!), the model converges to a *globally segregated* state — demonstrating how local preferences can amplify into global patterns.

In [ ]:
class SchellingModel:
    """Schelling segregation model on a square grid.
    
    Parameters
    ----------
    grid_size : int
        Side length of the square grid.
    n_agents : int
        Total number of agents (must be < grid_size**2).
    t : int
        Minimum number of same-type neighbours for an agent to be satisfied.
    frac_type1 : float
        Fraction of type-1 agents (type-2 makes up the rest).
    """
    EMPTY = 0; TYPE1 = 1; TYPE2 = 2

    def __init__(self, grid_size=30, n_agents=700, t=3, frac_type1=0.5, seed=RANDOM_SEED):
        rng = np.random.default_rng(seed)
        self.t = t
        self.size = grid_size

        # Populate grid
        cells = np.zeros(grid_size**2, dtype=int)
        n1 = int(n_agents * frac_type1)
        n2 = n_agents - n1
        cells[:n1] = self.TYPE1
        cells[n1:n1+n2] = self.TYPE2
        rng.shuffle(cells)
        self.grid = cells.reshape(grid_size, grid_size)
        self.rng = rng

    def _same_type_neighbors(self, r, c):
        """Count neighbors of the same type as cell (r, c)."""
        agent_type = self.grid[r, c]
        count = 0
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0:
                    continue
                nr, nc = (r + dr) % self.size, (c + dc) % self.size
                if self.grid[nr, nc] == agent_type:
                    count += 1
        return count

    def step(self):
        """One round: move all unsatisfied agents to random empty cells."""
        rows, cols = np.where(self.grid != self.EMPTY)
        order = self.rng.permutation(len(rows))
        empty_r, empty_c = np.where(self.grid == self.EMPTY)
        empty_cells = list(zip(empty_r, empty_c))
        self.rng.shuffle(empty_cells)

        moved = 0
        for idx in order:
            r, c = rows[idx], cols[idx]
            if self.grid[r, c] == self.EMPTY:
                continue   # was vacated in this round
            if self._same_type_neighbors(r, c) < self.t and empty_cells:
                nr, nc = empty_cells.pop()
                self.grid[nr, nc] = self.grid[r, c]
                self.grid[r, c]   = self.EMPTY
                empty_cells.append((r, c))
                moved += 1
        return moved

    def satisfaction_rate(self):
        """Fraction of agents that are satisfied."""
        rows, cols = np.where(self.grid != self.EMPTY)
        if len(rows) == 0:
            return 1.0
        satisfied = sum(
            1 for r, c in zip(rows, cols)
            if self._same_type_neighbors(r, c) >= self.t
        )
        return satisfied / len(rows)

    def plot_grid(self, ax, title=""):
        cmap = mcolors.ListedColormap(['#f0f0f0', 'steelblue', 'coral'])
        ax.imshow(self.grid, cmap=cmap, vmin=0, vmax=2, interpolation='nearest')
        ax.set_title(title); ax.axis('off')

print("SchellingModel class defined.")

### 3.2 Simulate: $t = 3$ (matching the NS05 slide example)

In [ ]:
# Parameters matching NS05 slides: t=3, ~70% occupancy on 30×30 grid
model = SchellingModel(grid_size=30, n_agents=700, t=3, seed=RANDOM_SEED)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
snapshots = [0, 5, 20, 50]
satisfaction_log = []

round_idx = 0
snap_idx  = 0

for snap in snapshots:
    while round_idx < snap:
        model.step()
        satisfaction_log.append(model.satisfaction_rate())
        round_idx += 1
    sat = model.satisfaction_rate()
    model.plot_grid(axes[snap_idx],
                    title=f"Round {snap}\nSatisfaction: {sat:.0%}")
    snap_idx += 1

plt.suptitle("Schelling Segregation Model ($t=3$, 30×30 grid)\n"
             "Blue = Type 1, Coral = Type 2, Grey = empty", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Plot satisfaction over rounds
fig, ax = plt.subplots(figsize=FIGURE_SIZE_SMALL)
ax.plot(satisfaction_log, color=NODE_COLOR, linewidth=1.8)
ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8)
ax.set_xlabel("Round")
ax.set_ylabel("Fraction of satisfied agents")
ax.set_title("Convergence of Schelling model ($t=3$)")
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Observation:** With $t = 3$, agents are perfectly happy being a minority (they only need 3 out of 8 same-type neighbours). Yet the model converges to a visually segregated state — reproducing the key result from the NS05 slides: *segregated outcomes are more likely even when agents do not desire segregation*.

### 3.3 Exercises

**Exercise 3 (guided):** Vary $t$ from 1 to 5 and run each model for 50 rounds. For each $t$:
1. Plot the final grid.
2. Compute the **segregation index**: fraction of each agent's same-type neighbours at convergence.

Do you observe the effect mentioned in NS05 slides — *if $t$ is larger, segregation is amplified*?

In [ ]:
t_values = [1, 2, 3, 4, 5]
n_rounds = 50

fig, axes = plt.subplots(1, len(t_values), figsize=(14, 3))

for ax, t in zip(axes, t_values):
    m = SchellingModel(grid_size=30, n_agents=700, t=t, seed=RANDOM_SEED)
    for _ in range(n_rounds):
        m.step()
    sat = m.satisfaction_rate()
    m.plot_grid(ax, title=f"$t={t}$\nSat: {sat:.0%}")

plt.suptitle("Effect of threshold $t$ on segregation (50 rounds)", y=1.02)
plt.tight_layout()
plt.show()

**Exercise 4 (open):** What happens when the two groups are unequal in size? Set `frac_type1 = 0.3` (30% Type 1, 70% Type 2) with $t = 3$.

1. Does the minority group form islands or disperse?
2. Compare the final satisfaction rates of Type 1 and Type 2 separately.

In [ ]:
# Exercise 4: your solution here
m_unequal = SchellingModel(grid_size=30, n_agents=700, t=3, frac_type1=0.3, seed=RANDOM_SEED)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
m_unequal.plot_grid(axes[0], title="Round 0 (unequal groups, t=3)")
for _ in range(50):
    m_unequal.step()
m_unequal.plot_grid(axes[1], title=f"Round 50\nSatisfaction: {m_unequal.satisfaction_rate():.0%}")
plt.tight_layout()
plt.show()

---
## Summary

Key formulas and methods introduced in this notebook:

| Concept | Formula / Method | Slide reference |
|---|---|---|
| Homophily test | Compare cross-type fraction to $2pq$ | NS05 |
| Average neighbor degree | $k_{nn}(i) = \frac{1}{k_i}\sum_j A_{ij}k_j$ | NS05 |
| Degree correlation function | $k_{nn}(k)$: slope → assortative/neutral/disassortative | NS05 |
| Assortativity coefficient | `nx.degree_assortativity_coefficient(G)` | NS05 |
| Schelling model | local threshold $t$ → global segregation | NS05 |

**Key insight:** Both homophily and structural balance (NS05) produce polarised, clustered social structures — homophily through similar nodes connecting, balance through sign-stable triangles. The Schelling model shows the same outcome through a spatial mechanism.